In [1]:
!pip install --user numpy torch transformers accelerate bitsandbytes


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
!pip install pillow

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
# Pixtral 12B Loader (A100-Optimized, No Quantization)

import sys, os

# Paths
SCRATCH = f"/scratch/{os.getenv('USER')}"
sys.path.insert(0, f"{SCRATCH}/pip_site")

import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

# Cache dirs
CACHE = f"{SCRATCH}/hf_cache"
os.makedirs(CACHE, exist_ok=True)
os.environ["HF_HOME"] = CACHE
os.environ["TRANSFORMERS_CACHE"] = CACHE
os.environ["HF_HUB_CACHE"] = CACHE
os.environ["HF_DATASETS_CACHE"] = CACHE

# GPU diagnostics
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

# Model configuration
MODEL = "mistral-community/pixtral-12b"

print("Loading Pixtral 12B...")
processor = AutoProcessor.from_pretrained(MODEL, cache_dir=CACHE)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL,
    torch_dtype=torch.bfloat16,    # A100 runs bfloat16 best
    device_map="cuda",             # put model fully on all detected GPUs
    cache_dir=CACHE,
    low_cpu_mem_usage=True
)

print("Pixtral 12B loaded.")

GPU: NVIDIA A100-PCIE-40GB
VRAM (GB): 42.405855232
Loading Pixtral 12B...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Pixtral 12B loaded.


In [5]:
!pip install pymupdf

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/cvmfs/hpc.rug.nl/versions/2023.01/rocky8/x86_64/intel/icelake/software/Python/3.10.4-GCCcore-11.3.0/bin/python -m pip install --upgrade pip' command.


In [3]:
# PNG loader for troubleshooting guides

import os
from pathlib import Path
from PIL import Image
import re

PNG_DIR = "/home5/p319166/TOCAPs/cleaned_png/"

def parse_filename(fname):
    """
    Extracts (tocap_id, page_number) from filenames like:
    DP17-TOCAP12_page_01.png
    """
    m = re.match(r"(.*)_page_(\d+)\.png", fname)
    if not m:
        return None, None
    tocap_id = m.group(1)
    page_num = int(m.group(2))
    return tocap_id, page_num


def load_all_pngs(png_dir=PNG_DIR):
    """
    Loads all PNG files and groups them by TOCAP identifier.
    Returns:
      {
        'DP17-TOCAP12': [
            {'page_num': 1, 'image': PIL.Image},
            {'page_num': 2, 'image': PIL.Image},
            ...
        ],
        ...
      }
    """
    png_dir = Path(png_dir)
    data = {}

    for img_path in png_dir.glob("*.png"):
        fname = img_path.name
        tocap_id, page_num = parse_filename(fname)
        if tocap_id is None:
            print(f"Skipping non-matching filename: {fname}")
            continue

        img = Image.open(img_path).convert("RGB")

        if tocap_id not in data:
            data[tocap_id] = []

        data[tocap_id].append({
            "page_num": page_num,
            "image": img
        })

    # Sort pages inside each TOCAP
    for key in data:
        data[key] = sorted(data[key], key=lambda x: x["page_num"])

    return data


# Load data
tocap_data = load_all_pngs()
print("Loaded TOCAPs:", list(tocap_data.keys()))

Loaded TOCAPs: ['DP17-TOCAP15', 'DP17-TOCAP26', 'DP17-TOCAP17', 'DP17-TOCAP4', 'DP17-TOCAP12', 'DP17-TOCAP6', 'DP17-TOCAP27', 'DP17-TOCAP16', 'DP17-TOCAP28', 'DP17-TOCAP7', 'DP17-TOCAP19', 'DP17-TOCAP18']


In [ ]:
# [DEPRECATED] Pixtral batch extractor (final)

import os
import json
import re
import traceback
import torch

SAVE_DIR = "/home5/p319166/pixtral_output"
os.makedirs(SAVE_DIR, exist_ok=True)

zero_shot_prompt = """
Extract procedural knowledge from this Dutch industrial troubleshooting diagram.
CRITICAL RULES:
Extract ONLY entities that ACTUALLY EXIST in the diagram - DO NOT invent or hallucinate entities
If you cannot see more entities clearly, STOP extracting and close the JSON properly
DO NOT repeat the same entity text multiple times with different IDs
Each entity should have UNIQUE text from a distinct node in the diagram
Use sequential IDs: E1, E2, E3, etc.
Types must be exactly: "Action", "Condition", or "Decision"
Preserve EXACT text from each node, including numbering (e.g., "0)", "4-0")
Capture EVERY arrow as an "isPreceededBy" relation
ENTITY TYPES:
Action: An operation to be performed (e.g., "Start Tocap 4", "Wissel product")
Condition: A condition or state to be verified (e.g., "Controleer product, maatvoering")
Decision: A decision point with branching outcomes (e.g., "4-0 Voldoet de kap aan Q productspecificatie's?")
RELATION TYPE:
isPreceededBy: The target step is preceded by the source step (arrow goes from source to target)
OUTPUT FORMAT (example with 3 entities - extract MORE if they exist):
{
"entities": [
{"id": "E1", "type": "Action", "text": "Start Tocap 4 Oppakken kap van bretslede"},
{"id": "E2", "type": "Decision", "text": "4-0 Voldoet de kap aan Q productspecificatie's?"},
{"id": "E3", "type": "Action", "text": "0) Wissel product"},
…
],
"relations": [
{"source": "E2", "target": "E1", "type": "isPreceededBy"},
{"source": "E3", "target": "E2", "type": "isPreceededBy"},
…
]
}
IMPORTANT: Extract all the entities you can see, quality over quantity. YOU MUST OUTPUT ONLY VALID JSON. NO EXPLANATIONS.
"""

# JSON extractor
def extract_json_from_text(text):
    m = re.search(r"```json\s*(\{.*?\})\s*```", text, flags=re.S)
    if m:
        try:
            return json.loads(m.group(1))
        except:
            pass

    first = text.find("{")
    last = text.rfind("}")
    if first != -1 and last != -1 and last > first:
        candidate = text[first:last+1]
        try:
            return json.loads(candidate)
        except:
            pass
    return None

# Generation wrapper
def safe_generate(processor, model, image, prompt_text, max_new_tokens=2048):

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt_text}
            ]
        }
    ]

    prompt = processor.apply_chat_template(
        messages,
        add_generation_prompt=True
    )

    inputs = processor(
        text=prompt,
        images=[image],
        return_tensors="pt"
    )

    dtype = next(model.parameters()).dtype

    # send to correct device
    inputs = {
        "input_ids": inputs["input_ids"].to(model.device),
        "attention_mask": inputs["attention_mask"].to(model.device),
        "pixel_values": inputs["pixel_values"].to(model.device, dtype=dtype)
    }

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0
        )

    return processor.decode(output_ids[0], skip_special_tokens=True)


# Main loop over TOCAPs
all_outputs = {}

for tocap_id, pages in tocap_data.items():
    print(f"Processing TOCAP: {tocap_id} ({len(pages)} pages)")
    tocap_results = []

    for page in pages:
        try:
            img = page["image"]
            page_num = page.get("page_num", page.get("page_number", None))

            raw_text = safe_generate(
                processor,
                model,
                img,
                zero_shot_prompt,
                max_new_tokens=2048
            )

            parsed = extract_json_from_text(raw_text)

            tocap_results.append({
                "page": page_num,
                "raw": raw_text,
                "json": parsed
            })

            print(f"  Page {page_num} ok")

        except Exception as e:
            print(f"  ERROR on page {page_num}: {e}")
            tocap_results.append({
                "page": page_num,
                "raw": None,
                "json": None,
                "error": str(e),
                "traceback": traceback.format_exc()
            })

    out_path = os.path.join(SAVE_DIR, f"{tocap_id}.json")
    with open(out_path, "w", encoding="utf8") as f:
        json.dump(tocap_results, f, ensure_ascii=False, indent=2)

    print(f"Saved {tocap_id} → {out_path}")
    all_outputs[tocap_id] = tocap_results

print("Done.")

In [4]:
# [DEPRECATED] Pixtral batch extractor (FIXED VERSION - handles incomplete JSON)

import os
import json
import re
import traceback
import torch

SAVE_DIR = "/home5/p319166/pixtral_outputs"
os.makedirs(SAVE_DIR, exist_ok=True)

zero_shot_prompt = """
Extract procedural knowledge from this Dutch industrial troubleshooting diagram.

CRITICAL RULES:

1. Extract ONLY entities that ACTUALLY EXIST in the diagram - DO NOT invent or hallucinate entities
2. If you cannot see more entities clearly, STOP extracting and close the JSON properly
3. DO NOT repeat the same entity text multiple times with different IDs
4. Each entity should have UNIQUE text from a distinct node in the diagram
5. Use sequential IDs: E1, E2, E3, etc.
6. Types must be exactly: "Action", "Condition", or "Decision"
7. Preserve EXACT text from each node, including numbering (e.g., "0)", "4-0")
8. Capture EVERY arrow as an "isPreceededBy" relation

ENTITY TYPES:

- Action: An operation to be performed (e.g., "Start Tocap 4", "Wissel product")
- Condition: A condition or state to be verified (e.g., "Controleer product, maatvoering")
- Decision: A decision point with branching outcomes (e.g., "4-0 Voldoet de kap aan Q productspecificatie's?")

RELATION TYPE:

- isPreceededBy: The target step is preceded by the source step (arrow goes from source to target)

EXPECTED VISUAL STRUCTURE:
- Rounded rectangles at top = Start of procedure
- Nodes with "ja/nee" branches = Decision points
- Rectangular boxes = Conditions to verify or Actions to perform
- Arrows show procedural flow (isPreceededBy relationships)
- Numbers in shapes indicate continuation to another page

EXAMPLES OF VISUAL STRUCTURE:
1. Rounded rectangle: {"id": "E1", "type": "Action", "text": "Start Tocap 4 Oppakken kap van bretslede"}
2. Decision node: {"id": "E2", "type": "Decision", "text": "4-0 Voldoet de kap aan Q productspecificatie's?"}
3. Arrow from E1 to E2: {"source": "E2", "target": "E1", "type": "isPreceededBy", "label": "next"}

OUTPUT FORMAT (example with 3 entities - extract MORE if they exist):
{
"entities": [
{"id": "E1", "type": "Action", "text": "Start Tocap 4 Oppakken kap van bretslede"},
{"id": "E2", "type": "Decision", "text": "4-0 Voldoet de kap aan Q productspecificatie's?"},
{"id": "E3", "type": "Action", "text": "0) Wissel product"},
…
],
"relations": [
{"source": "E2", "target": "E1", "type": "isPreceededBy"},
{"source": "E3", "target": "E2", "type": "isPreceededBy"},
…
]
}

IMPORTANT: Extract all the entities you can see, quality over quantity. YOU MUST OUTPUT ONLY VALID JSON. NO EXPLANATIONS."""


def extract_json_from_text(text):
    """
    Extract JSON from text, handling nested structures and incomplete JSON.
    """
    if not text:
        return None
    
    # Try to find JSON in markdown code block first
    json_match = re.search(r"\s*(\{.*?)\s*```", text, flags=re.DOTALL)
    if json_match:
        candidate = json_match.group(1)
        # Find the actual end of the JSON object
        brace_count = candidate.count("{") - candidate.count("}")
        
        # If braces are balanced, try to parse
        if brace_count == 0:
            try:
                return json.loads(candidate)
            except json.JSONDecodeError:
                pass
        elif brace_count > 0:
            # Need more closing braces - look after the match
            text_after = text[json_match.end(1):]
            for i, char in enumerate(text_after):
                if char == "}":
                    brace_count -= 1
                    if brace_count == 0:
                        candidate = candidate + text_after[:i+1]
                        try:
                            return json.loads(candidate)
                        except json.JSONDecodeError:
                            # Try to fix incomplete JSON
                            return try_fix_incomplete_json(candidate)
                elif char == "{":
                    brace_count += 1
            # If we never found the end, try to fix what we have
            return try_fix_incomplete_json(candidate)
    
    # Try to find JSON without markdown
    first_brace = text.find("{")
    if first_brace == -1:
        return None
    
    # Count braces to find matching closing brace
    brace_count = 0
    last_brace = -1
    
    for i in range(first_brace, len(text)):
        if text[i] == "{":
            brace_count += 1
        elif text[i] == "}":
            brace_count -= 1
            if brace_count == 0:
                last_brace = i
                break
    
    if last_brace > first_brace:
        candidate = text[first_brace:last_brace + 1]
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            return try_fix_incomplete_json(candidate)
    else:
        # Incomplete JSON - try to extract what we can
        candidate = text[first_brace:]
        return try_fix_incomplete_json(candidate)
    
    return None


def try_fix_incomplete_json(text):
    """
    Try to fix incomplete JSON by closing brackets and removing incomplete entries.
    """
    if not text or text.strip() == "":
        return None
    
    # Remove trailing incomplete items
    # Look for incomplete relations (missing closing brace)
    text = text.rstrip()
    
    # Try to close the JSON structure
    brace_count = text.count("{") - text.count("}")
    bracket_count = text.count("[") - text.count("]")
    
    # Close brackets first
    while bracket_count > 0:
        text += "]"
        bracket_count -= 1
    
    # Close braces
    while brace_count > 0:
        text += "}"
        brace_count -= 1
    
    # Remove trailing commas
    text = re.sub(r',\s*}', '}', text)
    text = re.sub(r',\s*]', ']', text)
    
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # Last resort: try to extract entities and relations separately
        return extract_partial_json(text)


def extract_partial_json(text):
    """
    Extract entities and relations even from incomplete JSON.
    """
    result = {"entities": [], "relations": []}
    
    # Try to find entities array
    entities_match = re.search(r'"entities"\s*:\s*\[(.*?)\]', text, flags=re.DOTALL)
    if entities_match:
        entities_text = entities_match.group(1)
        # Extract individual entity objects
        entity_pattern = r'\{\s*"id"\s*:\s*"([^"]+)"\s*,\s*"type"\s*:\s*"([^"]+)"\s*,\s*"text"\s*:\s*"([^"]+)"'
        for match in re.finditer(entity_pattern, entities_text):
            result["entities"].append({
                "id": match.group(1),
                "type": match.group(2),
                "text": match.group(3)
            })
    
    # Try to find relations array
    relations_match = re.search(r'"relations"\s*:\s*\[(.*?)\]', text, flags=re.DOTALL)
    if relations_match:
        relations_text = relations_match.group(1)
        # Extract individual relation objects
        relation_pattern = r'\{\s*"source"\s*:\s*"([^"]+)"\s*,\s*"target"\s*:\s*"([^"]+)"\s*,\s*"type"\s*:\s*"([^"]+)"'
        for match in re.finditer(relation_pattern, relations_text):
            result["relations"].append({
                "source": match.group(1),
                "target": match.group(2),
                "type": match.group(3)
            })
    
    return result if result["entities"] or result["relations"] else None


# Generation wrapper (FIXED - no prompt echo)
def safe_generate(processor, model, image, prompt_text, max_new_tokens=2048):
    """
    Generate response from image and prompt.
    Returns only the generated text (excludes prompt).
    """
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt_text}
            ]
        }
    ]
    
    prompt = processor.apply_chat_template(
        messages,
        add_generation_prompt=True
    )
    
    inputs = processor(
        text=prompt,
        images=[image],
        return_tensors="pt"
    )
    
    dtype = next(model.parameters()).dtype
    
    # Send to correct device
    inputs = {
        "input_ids": inputs["input_ids"].to(model.device),
        "attention_mask": inputs["attention_mask"].to(model.device),
        "pixel_values": inputs["pixel_values"].to(model.device, dtype=dtype)
    }
    
    # Store input length BEFORE generation
    input_length = inputs["input_ids"].shape[1]
    
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,  # ⚠️ Consider increasing this if JSON is cut off
            do_sample=False,
            temperature=0.0
        )
    
    # Extract only the newly generated tokens (skip input prompt)
    generated_ids = output_ids[0][input_length:]
    generated_text = processor.decode(generated_ids, skip_special_tokens=True)
    
    return generated_text.strip()


# Main loop over TOCAPs
all_outputs = {}

for tocap_id, pages in tocap_data.items():
    print(f"Processing TOCAP: {tocap_id} ({len(pages)} pages)")
    tocap_results = []
    
    for page in pages:
        try:
            img = page["image"]
            page_num = page.get("page_num", page.get("page_number", None))
            
            # ⚠️ Increase max_new_tokens if JSON is being cut off
            raw_text = safe_generate(
                processor,
                model,
                img,
                zero_shot_prompt,
                max_new_tokens=8196  # Increased from 2048
            )
            
            parsed = extract_json_from_text(raw_text)
            
            # Normalize relation format
            if parsed and "relations" in parsed:
                for relation in parsed["relations"]:
                    if "from" in relation and "source" not in relation:
                        relation["source"] = relation.pop("from")
                    if "to" in relation and "target" not in relation:
                        relation["target"] = relation.pop("to")
            
            tocap_results.append({
                "page": page_num,
                "raw": raw_text,
                "json": parsed
            })
            
            if parsed:
                entity_count = len(parsed.get("entities", []))
                relation_count = len(parsed.get("relations", []))
                print(f"  Page {page_num} ✓ ({entity_count} entities, {relation_count} relations)")
            else:
                print(f"  Page {page_num} ⚠ (JSON parse failed)")
                # Debug: show last 200 chars of raw text
                if raw_text:
                    print(f"    Last 200 chars: {raw_text[-200:]}")
            
        except Exception as e:
            print(f"  ERROR on page {page_num}: {e}")
            tocap_results.append({
                "page": page_num,
                "raw": None,
                "json": None,
                "error": str(e),
                "traceback": traceback.format_exc()
            })
    
    out_path = os.path.join(SAVE_DIR, f"{tocap_id}.json")
    with open(out_path, "w", encoding="utf8") as f:
        json.dump(tocap_results, f, ensure_ascii=False, indent=2)
    
    print(f"Saved {tocap_id} → {out_path}")
    all_outputs[tocap_id] = tocap_results

print("Done.")

Processing TOCAP: DP17-TOCAP15 (2 pages)


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


  Page 1 ✓ (176 entities, 0 relations)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


  Page 2 ✓ (250 entities, 0 relations)
Saved DP17-TOCAP15 → /home5/p319166/pixtral_outputs/DP17-TOCAP15.json
Processing TOCAP: DP17-TOCAP26 (2 pages)


...

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


  Page 2 ✓ (18 entities, 12 relations)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


  Page 3 ✓ (24 entities, 23 relations)
Saved DP17-TOCAP17 → /home5/p319166/pixtral_outputs/DP17-TOCAP17.json
Processing TOCAP: DP17-TOCAP4 (2 pages)


KeyboardInterrupt: 

In [ ]:
# Pixtral Complete Fixed Code - No Post-Processing Rescue
import os
import json
import re
import torch

SAVE_DIR = "/home5/p319166/pixtral_outputs"
os.makedirs(SAVE_DIR, exist_ok=True)

# IMPROVED PROMPT - Anti-hallucination
zero_shot_prompt = """
Extract procedural knowledge from this Dutch industrial troubleshooting diagram.

CRITICAL RULES:
Extract ONLY entities that ACTUALLY EXIST in the diagram - DO NOT invent or hallucinate entities
If you cannot see more entities clearly, STOP extracting and close the JSON properly
DO NOT repeat the same entity text multiple times with different IDs
Each entity should have UNIQUE text from a distinct node in the diagram
Use sequential IDs: E1, E2, E3, etc.
Types must be exactly: "Action", "Condition", or "Decision"
Preserve EXACT text from each node, including numbering (e.g., "0)", "4-0")
Capture EVERY arrow as an "isPreceededBy" relation

ENTITY TYPES:
Action: An operation to be performed (e.g., "Start Tocap 4", "Wissel product")
Condition: A condition or state to be verified (e.g., "Controleer product, maatvoering")
Decision: A decision point with branching outcomes (e.g., "4-0 Voldoet de kap aan Q productspecificatie's?")

RELATION TYPE:
isPreceededBy: The target step is preceded by the source step (arrow goes from source to target)

EXPECTED VISUAL STRUCTURE:
- Rounded rectangles at top = Start of procedure
- Nodes with "ja/nee" branches = Decision points
- Rectangular boxes = Conditions to verify or Actions to perform
- Arrows show procedural flow (isPreceededBy relationships)
- Numbers in shapes indicate continuation to another page

EXAMPLES OF VISUAL STRUCTURE:
1. Rounded rectangle: {"id": "E1", "type": "Action", "text": "Start Tocap 4 Oppakken kap van bretslede"}
2. Decision node: {"id": "E2", "type": "Decision", "text": "4-0 Voldoet de kap aan Q productspecificatie's?"}
3. Arrow from E1 to E2: {"source": "E2", "target": "E1", "type": "isPreceededBy", "label": "next"}

OUTPUT FORMAT (example with 3 entities - extract MORE if they exist):

{
"entities": [
{"id": "E1", "type": "Action", "text": "Start Tocap 4 Oppakken kap van bretslede"},
{"id": "E2", "type": "Decision", "text": "4-0 Voldoet de kap aan Q productspecificatie's?"},
{"id": "E3", "type": "Action", "text": "0) Wissel product"},
…
],
"relations": [
{"source": "E2", "target": "E1", "type": "isPreceededBy"},
{"source": "E3", "target": "E2", "type": "isPreceededBy"},
…
]
}

IMPORTANT: Extract all the entities you can see, quality over quantity. YOU MUST OUTPUT ONLY VALID JSON. NO EXPLANATIONS.
"""

def extract_json_from_text(text):
    """
    Extract JSON from text - NO RESCUE LOGIC.
    Returns parsed JSON or None if invalid.
    """
    if not text:
        return None
    
    # Remove markdown code blocks
    text = re.sub(r'```json\s*', '', text)
    text = re.sub(r'```\s*', '', text)
    text = text.strip()
    
    # Find JSON object
    first_brace = text.find("{")
    if first_brace == -1:
        return None
    
    # Count braces to find matching closing brace
    brace_count = 0
    last_brace = -1
    
    for i in range(first_brace, len(text)):
        if text[i] == "{":
            brace_count += 1
        elif text[i] == "}":
            brace_count -= 1
            if brace_count == 0:
                last_brace = i
                break
    
    if last_brace > first_brace:
        candidate = text[first_brace:last_brace + 1]
        try:
            result = json.loads(candidate)
            
            # Basic validation - must have both keys
            if "entities" in result and "relations" in result:
                return result
            else:
                print(f"    ⚠ JSON missing required keys")
                return None
                
        except json.JSONDecodeError as e:
            print(f"    ⚠ JSON parse error: {e}")
            return None
    
    print(f"    ⚠ Could not find complete JSON object")
    return None

def safe_generate(processor, model, image, prompt_text, max_new_tokens=16384):
    """
    Generate response from image and prompt.
    Returns only the generated text (excludes prompt).
    """
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt_text}
            ]
        }
    ]
    
    prompt = processor.apply_chat_template(
        messages,
        add_generation_prompt=True
    )
    
    inputs = processor(
        text=prompt,
        images=[image],
        return_tensors="pt"
    )
    
    dtype = next(model.parameters()).dtype
    
    # Send to correct device
    inputs = {
        "input_ids": inputs["input_ids"].to(model.device),
        "attention_mask": inputs["attention_mask"].to(model.device),
        "pixel_values": inputs["pixel_values"].to(model.device, dtype=dtype)
    }
    
    # Store input length BEFORE generation
    input_length = inputs["input_ids"].shape[1]
    
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=processor.tokenizer.pad_token_id
        )
    
    # Extract only the newly generated tokens (skip input prompt)
    generated_ids = output_ids[0][input_length:]
    generated_text = processor.decode(generated_ids, skip_special_tokens=True)
    
    return generated_text.strip()

# Main loop over TOCAPs
all_outputs = {}

for tocap_id, pages in tocap_data.items():
    print(f"\nProcessing TOCAP: {tocap_id} ({len(pages)} pages)")
    tocap_results = []
    
    for page in pages:
        try:
            img = page["image"]
            page_num = page.get("page_num", page.get("page_number", None))
            
            print(f"  Page {page_num}...", end=" ")
            
            # Generate with increased token limit
            raw_text = safe_generate(
                processor,
                model,
                img,
                zero_shot_prompt,
                max_new_tokens=16384 
            )
            
            # Parse JSON (no rescue)
            parsed = extract_json_from_text(raw_text)
            
            # Normalize relation format if parsed successfully
            if parsed and "relations" in parsed:
                for relation in parsed["relations"]:
                    if "from" in relation and "source" not in relation:
                        relation["source"] = relation.pop("from")
                    if "to" in relation and "target" not in relation:
                        relation["target"] = relation.pop("to")
            
            # Save result
            tocap_results.append({
                "page": page_num,
                "raw": raw_text,
                "json": parsed
            })
            
            if parsed:
                entity_count = len(parsed.get("entities", []))
                relation_count = len(parsed.get("relations", []))
                print(f"✓ ({entity_count} entities, {relation_count} relations)")
            else:
                print(f"✗ (JSON parse failed)")
            
        except Exception as e:
            print(f"✗ ERROR: {e}")
            tocap_results.append({
                "page": page_num,
                "raw": None,
                "json": None,
                "error": str(e)
            })
    
    # Save to file
    out_path = os.path.join(SAVE_DIR, f"{tocap_id}.json")
    with open(out_path, "w", encoding="utf8") as f:
        json.dump(tocap_results, f, ensure_ascii=False, indent=2)
    
    print(f"  Saved → {out_path}")
    all_outputs[tocap_id] = tocap_results

print("\n✓ All TOCAPs processed")

# Summary statistics
total_pages = sum(len(results) for results in all_outputs.values())
successful_pages = sum(1 for results in all_outputs.values() 
                       for r in results if r.get("json") is not None)
print(f"\nSummary: {successful_pages}/{total_pages} pages successfully extracted")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Processing TOCAP: DP17-TOCAP15 (2 pages)
  Page 1... 

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


    ⚠ Could not find complete JSON object
✗ (JSON parse failed)
  Page 2... 

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

    ⚠ Could not find complete JSON object
✗ (JSON parse failed)
  Page 2... 

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


✓ (21 entities, 29 relations)
  Saved → /home5/p319166/pixtral_outputs/DP17-TOCAP28.json

Processing TOCAP: DP17-TOCAP7 (2 pages)
  Page 1... 

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


    ⚠ Could not find complete JSON object
✗ (JSON parse failed)
  Page 2... 